# Set Up

In [1]:
# Import libraries
from google.colab import drive
import os
import json
import pandas as pd
import string
import numpy as np
import random
import networkx as nx
import matplotlib.pyplot as plt



In [2]:
# Mount drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [3]:
# Set assignment folder path
lab_folder = '/content/drive/MyDrive/Courses/INST414/Week 3'
os.chdir(lab_folder)

# Analysis

In [5]:
# Load dataset
df = pd.read_csv('spam.csv', encoding='latin-1')


In [6]:
# Clean up columns and display dataset
df = df[['v1', 'v2']]
df.columns = ['label', 'text']

display(df.head(10))

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
5,spam,FreeMsg Hey there darling it's been 3 week's n...
6,ham,Even my brother is not like to speak with me. ...
7,ham,As per your request 'Melle Melle (Oru Minnamin...
8,spam,WINNER!! As a valued network customer you have...
9,spam,Had your mobile 11 months or more? U R entitle...


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   5572 non-null   object
 1   text    5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


# Map labels to 0 and 1
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label_num'], test_size=0.2, random_state=42)

# Vectorize the text using TF-IDF (Term Frequency-Inverse Document Frequency)
vectorizer = TfidfVectorizer(stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train the Multinomial Naive Bayes model
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

MultinomialNB()

In [9]:
# Make predictions and evaluate
y_pred = model.predict(X_test_tfidf)
acc = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {acc:.4f}")
print("Confusion Matrix:")
print(conf_matrix)

Accuracy: 0.9668
Confusion Matrix:
[[965   0]
 [ 37 113]]


In [10]:
# Find the prediction errors: align predictions with the original text in the test set
results_df = pd.DataFrame({
    'text': X_test,
    'actual': y_test,
    'predicted': y_pred
})

# Filter where the prediction does not match the actual label
errors_df = results_df[results_df['actual'] != results_df['predicted']].copy()
errors_df['actual_label'] = errors_df['actual'].map({0: 'Ham (Legit)', 1: 'Spam'})
errors_df['predicted_label'] = errors_df['predicted'].map({0: 'Ham (Legit)', 1: 'Spam'})

# Grab 5 samples
five_errors = errors_df.head(5)
print("\n--- 5 Samples the Model Got Wrong ---")
for index, row in five_errors.iterrows():
    print(f"Text: {row['text']}")
    print(f"Actual: {row['actual_label']} | Predicted: {row['predicted_label']}")
    print("-" * 50)


--- 5 Samples the Model Got Wrong ---
Text: We know someone who you know that fancies you. Call 09058097218 to find out who. POBox 6, LS15HB 150p
Actual: Spam | Predicted: Ham (Legit)
--------------------------------------------------
Text: Hi I'm sue. I am 20 years old and work as a lapdancer. I love sex. Text me live - I'm i my bedroom now. text SUE to 89555. By TextOperator G2 1DA 150ppmsg 18+
Actual: Spam | Predicted: Ham (Legit)
--------------------------------------------------
Text: Loans for any purpose even if you have Bad Credit! Tenants Welcome. Call NoWorriesLoans.com on 08717111821
Actual: Spam | Predicted: Ham (Legit)
--------------------------------------------------
Text: tddnewsletter@emc1.co.uk (More games from TheDailyDraw) Dear Helen, Dozens of Free Games - with great prizesWith..
Actual: Spam | Predicted: Ham (Legit)
--------------------------------------------------
Text: ringtoneking 84484
Actual: Spam | Predicted: Ham (Legit)
-----------------------------------